# Graphs and uninformed search: notebook

The room of RO-009 as a search problem: its 0.5 m grid (14 x 8 = 112 cells, 36 blocked, 76 free) and a small
waypoint graph of the same room. The geometry lives in `images/roommap.py`, the searches in `images/roomsearch.py`;
here we check every number the Note quotes. Every search is deterministic, so no random seed is needed.

In [1]:
import sys, math, itertools
sys.path.insert(0, 'images')
import roomsearch as rs
from roommap import ROBOT, TABLE_F

## 1. The opening problem (Note, Section 1)

In [2]:
X, Y, TH = ROBOT
a, b, c = TABLE_F[3]                                   # the table's left edge: -4x + y + 11 = 0
hit = -(a * X + b * Y + c) / (a * math.cos(TH) + b * math.sin(TH))
print(f'driving straight ahead at 30 degrees, the centre meets the table after {hit:.2f} m')
print('cells:', rs.NX * rs.NY, ' blocked:', rs.BLOCKED.sum(), ' free:', len(rs.FREE))
print('start cell', rs.START, 'x', rs.START[0] * 0.5, '-', rs.START[0] * 0.5 + 0.5, ' goal cell', rs.GOAL)

driving straight ahead at 30 degrees, the centre meets the table after 1.35 m
cells: 112  blocked: 36  free: 76
start cell (4, 2) x 2.0 - 2.5  goal cell (7, 7)


## 2. The grid as a graph, and the waypoint graph (Note, Section 2)

In [3]:
edges = sorted({tuple(sorted((c, n))) for c in rs.FREE for n, _ in rs.neighbours(c)})
print('vertices:', len(rs.FREE), ' edges:', len(edges))
print('neighbours of the start cell:', [n for n, _ in rs.neighbours(rs.START)])
print('waypoints:', rs.WP)
for a, b in rs.WP_EDGES:
    print(f'  {a}-{b}: {rs.wp_dist(a, b):.3f} m')
print('edges:', len(rs.WP_EDGES), ' straight line S to the door:', round(rs.wp_dist('S', 'G'), 3), 'm (over the 2.8 m limit)')

vertices: 76  edges: 114
neighbours of the start cell: [(3, 2), (4, 1), (4, 3), (5, 2)]
waypoints: {'S': (2.0, 1.0), 'A': (3.0, 1.0), 'B': (5.0, 1.0), 'C': (5.5, 2.5), 'D': (3.5, 3.0), 'P': (1.0, 3.0), 'G': (3.6, 4.0)}
  A-B: 2.000 m
  A-D: 2.062 m
  A-S: 1.000 m
  B-C: 1.581 m
  C-D: 2.062 m
  C-G: 2.421 m
  D-G: 1.005 m
  D-P: 2.500 m
  D-S: 2.500 m
  G-P: 2.786 m
  P-S: 2.236 m
edges: 11  straight line S to the door: 3.4 m (over the 2.8 m limit)


## 3. The search tree of the waypoint graph (Note, Section 3)

In [4]:
T = rs.tree_paths(rs.wp_nbrs, 'S', 'G', 6)
levels = [sum(len(p) == k + 1 for p in T) for k in range(7)]
print('paths per level:', levels, ' total:', len(T), ' reaching G:', sum(p[-1] == 'G' for p in T))
T3 = [p for p in T if len(p) <= 4]
print('levels 0-3:', len(T3), 'paths,', sum(p[-1] == 'G' for p in T3), 'reach G; copies of D:', sum(p[-1] == 'D' for p in T3))
print('paths that reach G:', ['-'.join(p) for p in T if p[-1] == 'G'])

paths per level: [1, 3, 8, 11, 10, 5, 2]  total: 40  reaching G: 14
levels 0-3: 23 paths, 6 reach G; copies of D: 3
paths that reach G: ['S-D-G', 'S-P-G', 'S-A-D-G', 'S-D-C-G', 'S-D-P-G', 'S-P-D-G', 'S-A-B-C-G', 'S-A-D-C-G', 'S-A-D-P-G', 'S-P-D-C-G', 'S-A-B-C-D-G', 'S-D-A-B-C-G', 'S-A-B-C-D-P-G', 'S-P-D-A-B-C-G']


## 4. The search loop with a visited list, on the waypoint graph (Note, Section 4)

In [5]:
r = rs.bfs(start='S', goal='G', nbrs=rs.wp_nbrs)
for k, s in enumerate(r['snaps'], 1):
    print(f"expansion {k}: take {s['cur']}; queue {s['frontier']}; visited {s['visited']}")
print('route:', r['path'])

expansion 1: take S; queue ['A', 'D', 'P']; visited ['S']
expansion 2: take A; queue ['D', 'P', 'B']; visited ['S', 'A']
expansion 3: take D; queue ['P', 'B', 'C', 'G']; visited ['S', 'A', 'D']
expansion 4: take P; queue ['B', 'C', 'G']; visited ['S', 'A', 'D', 'P']
expansion 5: take B; queue ['C', 'G']; visited ['S', 'A', 'D', 'P', 'B']
expansion 6: take C; queue ['G']; visited ['S', 'A', 'D', 'P', 'B', 'C']
expansion 7: take G; queue []; visited ['S', 'A', 'D', 'P', 'B', 'C', 'G']
route: ['S', 'D', 'G']


## 5. Breadth-first search on the grid (Note, Section 5)

In [6]:
r = rs.bfs()
print('moves:', len(r['path']) - 1, ' length:', rs.length(r['path']), 'm  expansions:', len(r['visited']))
print('route:', r['path'])
full = rs.bfs(goal=None)
ring = {rs.START: 0}
for c in full['visited']:
    if full['parent'][c] is not None:
        ring[c] = ring[full['parent'][c]] + 1
print('cells reached without stopping at G:', len(full['visited']))
print('ring sizes:', [sum(v == k for v in ring.values()) for k in range(max(ring.values()) + 1)])
print('largest queue:', max(len(s['frontier']) for s in r['snaps']))

moves: 8  length: 4.0 m  expansions: 57
route: [(4, 2), (4, 3), (4, 4), (4, 5), (4, 6), (4, 7), (5, 7), (6, 7), (7, 7)]
cells reached without stopping at G: 76
ring sizes: [1, 4, 7, 9, 10, 9, 7, 6, 5, 4, 3, 2, 1, 1, 2, 2, 1, 1, 1]
largest queue: 11


## 6. Depth-first search on the grid (Note, Section 6)

In [7]:
d = rs.dfs()
print('moves:', len(d['path']) - 1, ' length:', rs.length(d['path']), 'm  expansions:', len(d['visited']),
      ' dead ends:', sum(s['dead'] for s in d['snaps']))
print('largest stack:', max(len(s['frontier']) for s in d['snaps']))
# every one of the 24 orders in which the four moves can be tried
res = []
for order in itertools.permutations(rs.STEPS4):
    db = rs.dfs(steps=list(order)); bb = rs.bfs(steps=list(order))
    res.append((len(db['path']) - 1, len(bb['path']) - 1, len(db['visited']), len(bb['visited'])))
print('DFS over all 24 orders: route from', min(r[0] for r in res), 'to', max(r[0] for r in res), 'moves;',
      'expansions from', min(r[2] for r in res), 'to', max(r[2] for r in res))
print('BFS over all 24 orders: route', sorted({r[1] for r in res}), 'moves; expansions from',
      min(r[3] for r in res), 'to', max(r[3] for r in res))

moves: 22  length: 11.0 m  expansions: 55  dead ends: 8
largest stack: 15
DFS over all 24 orders: route from 8 to 26 moves; expansions from 9 to 65
BFS over all 24 orders: route [8] moves; expansions from 54 to 58


The Python box of Section 6: the same loop gives both searches; only the end of the frontier the next cell
comes from differs (and the stack takes the new cells in reverse, so the first neighbour is tried first).

In [8]:
from collections import deque

def search(start, goal, neighbours, depth_first=False):
    frontier, parent = deque([start]), {start: None}
    while frontier:
        cell = frontier.pop() if depth_first else frontier.popleft()
        if cell == goal:
            break
        new = [n for n in neighbours(cell) if n not in parent]
        for n in (reversed(new) if depth_first else new):
            parent[n] = cell
            frontier.append(n)
    route = [goal]
    while parent[route[-1]] is not None:
        route.append(parent[route[-1]])
    return route[::-1]

nbrs = lambda c: [n for n, _ in rs.neighbours(c)]
print(len(search(rs.START, rs.GOAL, nbrs)) - 1)
print(len(search(rs.START, rs.GOAL, nbrs, depth_first=True)) - 1)

8
22


## 7. How the work grows (Note, Section 7)

In [9]:
P = rs.tree_paths(lambda c: rs.neighbours(c), rs.START, None, 8)
room = [sum(len(p) == d + 1 for p in P) for d in range(9)]
four = [4 ** d for d in range(9)]
print('4^d:', four, ' sum:', sum(four))
print('room paths per level:', room, ' sum:', sum(room))
print('level-8 room paths ending at the door:', sum(len(p) == 9 and p[-1] == rs.GOAL for p in P))
print('geometric sum check, b = 4, d = 2:', 1 + 4 + 16, (4 ** 3 - 1) // 3)
print('BFS work on the grid: expansions + neighbour checks =', 76, '+', 2 * len(edges), '=', 76 + 2 * len(edges))
ops = 1e8                                                           # basic steps per second
for name, n in [('4^20', 4 ** 20), ('18^10', 18 ** 10), ('18^20', 18 ** 20)]:
    print(f'{name} = {n:.3g}; at 1e8 per second: {n / ops:.3g} s = {n / ops / 3600:.3g} h = {n / ops / 3.156e7:.3g} years')

4^d: [1, 4, 16, 64, 256, 1024, 4096, 16384, 65536]  sum: 87381
room paths per level: [1, 4, 11, 30, 69, 158, 342, 728, 1447]  sum: 2790
level-8 room paths ending at the door: 16
geometric sum check, b = 4, d = 2: 21 21
BFS work on the grid: expansions + neighbour checks = 76 + 228 = 304
4^20 = 1.1e+12; at 1e8 per second: 1.1e+04 s = 3.05 h = 0.000348 years
18^10 = 3.57e+12; at 1e8 per second: 3.57e+04 s = 9.92 h = 0.00113 years
18^20 = 1.27e+25; at 1e8 per second: 1.27e+17 s = 3.54e+13 h = 4.04e+09 years


## 8. Grids in more dimensions (Note, Section 8)

In [10]:
print('x, y:', 14 * 8, ' x, y, 8 headings:', 14 * 8 * 8, ' 6 joints, 100 values each:', f'{100 ** 6:.0e}')
print('seconds at 1e8 cells per second:', 100 ** 6 / 1e8, '=', round(100 ** 6 / 1e8 / 3600, 1), 'h')

x, y: 112  x, y, 8 headings: 896  6 joints, 100 values each: 1e+12
seconds at 1e8 cells per second: 10000.0 = 2.8 h


## 9. Explicit or implicit (Note, Section 9)

In [11]:
print('20 m x 20 m, 0.25 m cells, 8 headings:', 80 * 80 * 8)
print('200 m x 200 m, 0.25 m cells, 16 headings:', 800 * 800 * 16)

20 m x 20 m, 0.25 m cells, 8 headings: 51200
200 m x 200 m, 0.25 m cells, 16 headings: 10240000


## 10. A battery in the state (Note, Section 10)

In [12]:
A = rs.dfs_battery(12, full_state=False)
B = rs.dfs_battery(12, full_state=True)
first = dict(A['visited'])
print('state = cell: route found?', A['path'] is not None, ' expansions:', len(A['visited']))
print('  charge left when the cell (4, 4) was first reached:', first[(4, 4)])
print('state = (cell, charge):', len(B['path']) - 1, 'moves,', len(B['visited']), 'expansions')
print('number of (cell, charge) states:', len(rs.FREE) * 13)
# breadth-first search with the cell alone as the state and the same battery: it reaches every cell first by the
# fewest moves, so with the most charge left
from collections import deque
parent, q, charge = {rs.START: None}, deque([rs.START]), {rs.START: 12}
while q:
    c = q.popleft()
    if c == rs.GOAL or charge[c] == 0:
        continue
    for n, _ in rs.neighbours(c):
        if n not in parent:
            parent[n], charge[n] = c, charge[c] - 1
            q.append(n)
print('BFS, state = cell, battery 12: door reached with', charge[rs.GOAL], 'units left, after', 12 - charge[rs.GOAL], 'moves')

state = cell: route found? False  expansions: 42
  charge left when the cell (4, 4) was first reached: 2
state = (cell, charge): 12 moves, 182 expansions
number of (cell, charge) states: 988
BFS, state = cell, battery 12: door reached with 4 units left, after 8 moves
